# Model Comparison: MovieLens 1M Recommenders

This notebook compares multiple recommender models on the MovieLens 1M dataset
using the same time-based train/val/test split and evaluation metrics.

Models evaluated:
- Random (seed=42)
- Popularity Baseline
- Mean Rating Baseline
- Item-Item KNN (collaborative filtering)
- SVD (latent factors)
- Content-Based (cosine similarity)
- Hybrid (alpha tuned on validation set)

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is in path
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)  # models use relative data/ paths

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.baseline import (
    MeanRatingBaseline,
    PopularityBaseline,
    RandomRecommender,
    evaluate,
    time_based_train_val_test_split,
)
from src.models.collaborative import ItemItemKNNRecommender, SVDRecommender
from src.models.content_based import ContentBasedRecommender
from src.models.hybrid import (
    CFHybridRecommender,
    HybridRecommender,
    tune_alpha_on_val,
    tune_cf_weights_on_val,
)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Load data
ratings_cols = ["user_id", "movie_id", "rating", "timestamp"]
data_dir = project_root / "data" / "ml-1m"
ratings = pd.read_csv(
    data_dir / "ratings.dat",
    sep="::",
    names=ratings_cols,
    engine="python",
    encoding="latin-1",
)
print(f"Loaded {len(ratings)} ratings from {ratings['user_id'].nunique()} users and {ratings['movie_id'].nunique()} movies")

# Time-based train/val/test split
train, val, test = time_based_train_val_test_split(
    ratings, val_ratio=0.1, test_ratio=0.2
)
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

## 1. Baseline Models

In [ ]:
# Random Baseline
print("--- Random Baseline ---")
rand_model = RandomRecommender(seed=42)
rand_model.fit(train)
rand_metrics = evaluate(rand_model, train, test, k=10)
print(f"Precision@10: {rand_metrics['precision_at_10']:.4f}")
print(f"Recall@10:    {rand_metrics['recall_at_10']:.4f}")

# Popularity Baseline
print("\n--- Popularity Baseline ---")
pop_model = PopularityBaseline()
pop_model.fit(train)
pop_metrics = evaluate(pop_model, train, test, k=10)
print(f"Precision@10: {pop_metrics['precision_at_10']:.4f}")
print(f"Recall@10:    {pop_metrics['recall_at_10']:.4f}")

# Mean Rating Baseline
print("\n--- Mean Rating Baseline ---")
mean_model = MeanRatingBaseline(min_ratings=5)
mean_model.fit(train)
mean_metrics = evaluate(mean_model, train, test, k=10)
print(f"Precision@10: {mean_metrics['precision_at_10']:.4f}")
print(f"Recall@10:    {mean_metrics['recall_at_10']:.4f}")

## 2. Content-Based Model

In [ ]:
print("--- Content-Based ---")
cb_model = ContentBasedRecommender(min_rating_for_like=4.0)
cb_model.fit(train)
cb_metrics = evaluate(cb_model, train, test, k=10)
print(f"Precision@10: {cb_metrics['precision_at_10']:.4f}")
print(f"Recall@10:    {cb_metrics['recall_at_10']:.4f}")

## 3. Hybrid Model - Alpha Tuning on Validation Set

In [ ]:
print("--- Tuning alpha on validation set ---")
alphas = [round(i * 0.1, 1) for i in range(11)]  # 0.0 to 1.0
best_alpha, val_results = tune_alpha_on_val(train, val, alphas=alphas, k=10)

print("\nValidation curve (alpha, P@10, R@10):")
for alpha in alphas:
    r = val_results[alpha]
    marker = " *" if alpha == best_alpha else ""
    print(f"  alpha={alpha:.1f}: P@10={r['precision_at_10']:.4f}, R@10={r['recall_at_10']:.4f}{marker}")

In [ ]:
# Plot validation curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

alphas = [round(i * 0.1, 1) for i in range(11)]
precisions = [val_results[a]["precision_at_10"] for a in alphas]
recalls = [val_results[a]["recall_at_10"] for a in alphas]

# Precision@10
axes[0].plot(alphas, precisions, "o-", color="steelblue", label="Precision@10")
axes[0].axvline(best_alpha, color="red", linestyle="--", label=f"Best α={best_alpha}")
axes[0].set_xlabel("Alpha")
axes[0].set_ylabel("Precision@10")
axes[0].set_title("Validation: Precision@10 vs Alpha")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Recall@10
axes[1].plot(alphas, recalls, "o-", color="coral", label="Recall@10")
axes[1].axvline(best_alpha, color="red", linestyle="--", label=f"Best α={best_alpha}")
axes[1].set_xlabel("Alpha")
axes[1].set_ylabel("Recall@10")
axes[1].set_title("Validation: Recall@10 vs Alpha")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Best Hybrid Model on Test Set

In [ ]:
print(f"--- Best Hybrid (alpha={best_alpha}) on Test Set ---")
best_hybrid = HybridRecommender(alpha=best_alpha)
best_hybrid.fit(train, data_dir=str(data_dir))
hybrid_metrics = evaluate(best_hybrid, train, test, k=10)
print(f"Precision@10: {hybrid_metrics['precision_at_10']:.4f}")
print(f"Recall@10:    {hybrid_metrics['recall_at_10']:.4f}")

## 4. Collaborative Filtering Models

Item-item KNN over co-ratings and truncated SVD latent factors.

Note: this notebook uses the 3-way train/val/test split (80/10/10),
so numbers differ slightly from the 80/20 split in `scripts/benchmark.py`.

In [ ]:
print('--- Item-Item KNN ---')
knn_model = ItemItemKNNRecommender(n_neighbors=50)
knn_model.fit(train)
knn_metrics = evaluate(knn_model, train, test, k=10)
print(f'Precision@10: {knn_metrics["precision_at_10"]:.4f}')
print(f'Recall@10:    {knn_metrics["recall_at_10"]:.4f}')

print('\n--- SVD (50 factors) ---')
svd_model = SVDRecommender(n_factors=50)
svd_model.fit(train)
svd_metrics = evaluate(svd_model, train, test, k=10)
print(f'Precision@10: {svd_metrics["precision_at_10"]:.4f}')
print(f'Recall@10:    {svd_metrics["recall_at_10"]:.4f}')

## 5. Three-Way CF Hybrid (CF + content + popularity)

Weights are tuned on the validation split; components are fitted once
and the grid reuses cached per-user score vectors.

In [ ]:
print('--- Tuning CF hybrid weights on validation ---')
best_weights, weight_results = tune_cf_weights_on_val(train, val, n_val_users=800)
print(f'Best weights: cf={best_weights[0]}, content={best_weights[1]}')

print(f'\n--- CFHybrid (cf={best_weights[0]}, content={best_weights[1]}) on Test ---')
cfhybrid_model = CFHybridRecommender(
    cf_weight=best_weights[0], content_weight=best_weights[1]
)
cfhybrid_model.fit(train, data_dir=str(data_dir))
cfhybrid_metrics = evaluate(cfhybrid_model, train, test, k=10)
print(f'Precision@10: {cfhybrid_metrics["precision_at_10"]:.4f}')
print(f'Recall@10:    {cfhybrid_metrics["recall_at_10"]:.4f}')

## 7. Comparison Table

In [ ]:
# Build comparison table
comparison = pd.DataFrame({
    "Model": [
        "Random",
        "Popularity",
        "MeanRating",
        "ItemItemKNN",
        "SVD",
        "CFHybrid",
        "ContentBased",
        f"Hybrid (α={best_alpha})",
    ],
    "Precision@10": [
        rand_metrics["precision_at_10"],
        pop_metrics["precision_at_10"],
        mean_metrics["precision_at_10"],
        knn_metrics["precision_at_10"],
        svd_metrics["precision_at_10"],
        cfhybrid_metrics["precision_at_10"],
        cb_metrics["precision_at_10"],
        hybrid_metrics["precision_at_10"],
    ],
    "Recall@10": [
        rand_metrics["recall_at_10"],
        pop_metrics["recall_at_10"],
        mean_metrics["recall_at_10"],
        knn_metrics["recall_at_10"],
        svd_metrics["recall_at_10"],
        cfhybrid_metrics["recall_at_10"],
        cb_metrics["recall_at_10"],
        hybrid_metrics["recall_at_10"],
    ],
})

comparison["P@10 Gain vs Pop"] = comparison["Precision@10"] - comparison.loc[1, "Precision@10"]
comparison["R@10 Gain vs Pop"] = comparison["Recall@10"] - comparison.loc[1, "Recall@10"]

print(comparison.to_string(index=False))

In [ ]:
# Plot comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = comparison["Model"].tolist()
p10 = comparison["Precision@10"].tolist()
r10 = comparison["Recall@10"].tolist()
colors = ["gray", "steelblue", "lightblue", "green", "purple", "teal", "orange", "red"]

axes[0].bar(models, p10, color=colors, edgecolor="black")
axes[0].set_ylabel("Precision@10")
axes[0].set_title("Model Comparison: Precision@10")
axes[0].tick_params(axis="x", rotation=15)
for i, v in enumerate(p10):
    axes[0].text(i, v + 0.001, f"{v:.4f}", ha="center", va="bottom")

axes[1].bar(models, r10, color=colors, edgecolor="black")
axes[1].set_ylabel("Recall@10")
axes[1].set_title("Model Comparison: Recall@10")
axes[1].tick_params(axis="x", rotation=15)
for i, v in enumerate(r10):
    axes[1].text(i, v + 0.001, f"{v:.4f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
print("=" * 60)
print("MOVIELENS 1M - MODEL COMPARISON SUMMARY")
print("=" * 60)
print(f"Best Hybrid alpha: {best_alpha}")
print(f"Validation P@10 at best alpha: {val_results[best_alpha]['precision_at_10']:.4f}")
print(f"Test P@10 (best hybrid): {hybrid_metrics['precision_at_10']:.4f}")
print(f"Test R@10 (best hybrid): {hybrid_metrics['recall_at_10']:.4f}")
print()
print("Key findings:")
print(f"  - Hybrid (α={best_alpha}) outperforms all baselines on Precision@10")
print(f"  - Hybrid gain vs Popularity: +{hybrid_metrics['precision_at_10'] - pop_metrics['precision_at_10']:.4f} P@10")
print("  - Content-Based alone is weaker than Popularity baseline")
print("  - Popularity is a strong baseline for this dataset")
print("=" * 60)